In [ ]:
#setup
!pip install torch tiktoken tensorflow tqdm numpy pandas psutil


In [ ]:
from importlib_metadata import version
print(version('torch'))
print(version('tiktoken'))

2.10.0+cpu
0.12.0


In [ ]:
import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)

with open("the-verdict.txt","r") as f:
    raw_text = f.read()

print(len(raw_text))
print(raw_text[:1000])

20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)

"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it's going to send the value of my picture 'way up; but I don't think of that, Mr. Rickham--the loss to Arrt is all I think of." The word, on Mrs. Thwing's lips, multiplied its _rs_ as though they were reflected in an endless vista of mirrors. And it was not only the Mrs. Thwings who mourned. Had not the exquisite Hermia Croft, at the last Grafton Gallery show, stopped me before Gisburn's "Moon-dancers" to say, with tears in her eyes: "We shall not look upon its like again"?

Well!--even thro

In [ ]:
#do some basic text splitting
import re
regex_pattern = r"\W+"
split_text = re.split(regex_pattern, raw_text)
split_text = [w.lower() for w in split_text if w]
print(f'Total tokens: {len(split_text)}')
print(split_text[:100])

Total tokens: 3826
['i', 'had', 'always', 'thought', 'jack', 'gisburn', 'rather', 'a', 'cheap', 'genius', 'though', 'a', 'good', 'fellow', 'enough', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', 'in', 'the', 'height', 'of', 'his', 'glory', 'he', 'had', 'dropped', 'his', 'painting', 'married', 'a', 'rich', 'widow', 'and', 'established', 'himself', 'in', 'a', 'villa', 'on', 'the', 'riviera', 'though', 'i', 'rather', 'thought', 'it', 'would', 'have', 'been', 'rome', 'or', 'florence', 'the', 'height', 'of', 'his', 'glory', 'that', 'was', 'what', 'the', 'women', 'called', 'it', 'i', 'can', 'hear', 'mrs', 'gideon', 'thwing', 'his', 'last', 'chicago', 'sitter', 'deploring', 'his', 'unaccountable', 'abdication', 'of', 'course', 'it', 's', 'going', 'to', 'send', 'the', 'value', 'of', 'my', 'picture', 'way']


In [ ]:
#create a set of words
all_words: set[str] = sorted(set(split_text))
all_words.extend(['<unk>'])
print(f'Total unique words: {len(all_words)}')
print(all_words[:100])

Total unique words: 1085
['_am_', '_famille', '_felt_', '_has_', '_have_', '_i', '_jardiniere_', '_mine_', '_not_', '_rose', '_rs_', '_that_', '_the_', '_was_', '_were_', 'a', 'abdication', 'able', 'about', 'above', 'abruptly', 'absolute', 'absorbed', 'absurdity', 'academic', 'accuse', 'accustomed', 'across', 'activity', 'add', 'added', 'admirers', 'adopted', 'adulation', 'advance', 'aesthetic', 'affect', 'afraid', 'after', 'afterward', 'again', 'ago', 'ah', 'air', 'alive', 'all', 'almost', 'alone', 'along', 'always', 'amazement', 'amid', 'among', 'amplest', 'amusing', 'an', 'and', 'another', 'answer', 'answered', 'any', 'anything', 'anywhere', 'apparent', 'apparently', 'appearance', 'appeared', 'appointed', 'are', 'arm', 'arms', 'arrt', 'art', 'articles', 'artist', 'as', 'aside', 'asked', 'at', 'atmosphere', 'atom', 'attack', 'attention', 'attitude', 'audacities', 'away', 'awful', 'axioms', 'azaleas', 'back', 'background', 'balance', 'balancing', 'balustraded', 'basking', 'bath', 'be'

In [ ]:
#create a vocab map word->int
vocab: dict[str|int] = {
    word:idx for idx,word in enumerate(all_words)
}

In [ ]:
#pass in input string, get output int tokens, assuming that we created a vocab set(unique words)

class Tokenizer:
  def __init__(self,vocab):
    self.str_to_int = vocab
    self.int_to_str = {
        value:key for key,value in vocab.items()
    }
    self.unk = vocab.get('<unk>')

  def encode(self, text:str) -> list[int]:
    if not isinstance(text, str):
      raise ValueError("Input must be a string")

    tokens = []
    regex_pattern = r"\W+"
    words = re.split(regex_pattern, text)
    words = [w.lower() for w in words if w]
    tokens = [self.str_to_int.get(word,self.unk) for word in words]
    return tokens

  def decode(self, tokens:list[int]) -> str:
    if not isinstance(tokens, list):
      raise ValueError("Input must be a list of integers")
    text = " ".join(self.int_to_str.get(t,"<unk>") for t in tokens)
    return text


tk = Tokenizer(vocab)
print(tk.encode("begin world"))
print(tk.decode([109,1084]))

[109, 1084]
begin <unk>
